## Predicting Stock Market Prices	
Develop dynamic and interactive candlestick chart visualizations using Plotly to represent TSLA stock price movements, incorporating hover details and zoom functionality for granular data exploration.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

In [2]:
data = pd.read_csv(
    "TSLA_cleaned_data.csv",
    parse_dates=["Date"]
)

print("Dataset Shape:", data.shape)
display(data.head())

Dataset Shape: (100, 6)


,Date,Open,High,Low,Close,Volume
0,2025-04-29,402.54,404.82,400.82,402.89,96503640
1,2025-03-18,414.37,417.58,406.36,411.17,129923902
2,2025-04-10,406.52,410.38,403.81,409.23,113709729
3,2025-03-06,405.40,405.98,399.78,404.15,119732002
4,2025-03-05,411.49,415.52,399.13,402.75,48754535


In [3]:
required_columns = [
    "Date", "Open", "High", "Low", "Close", "Volume"
]

missing_columns = [
    col for col in required_columns
    if col not in data.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

for col in ["Open", "High", "Low", "Close", "Volume"]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

data = data.dropna(subset=required_columns)

# Remove duplicate dates for charting
data = data.drop_duplicates(subset=["Date"], keep="first")

data = data.sort_values("Date").reset_index(drop=True)

print("Prepared Dataset:", data.shape)
display(data.head())

Prepared Dataset: (95, 6)


,Date,Open,High,Low,Close,Volume
0,2025-01-02,377.53,386.12,374.68,381.88,97077442
1,2025-01-03,372.39,377.55,371.43,376.36,85746412
2,2025-01-06,377.69,381.99,373.44,380.68,56893889
3,2025-01-07,387.26,390.45,385.32,386.06,76990555
4,2025-01-08,372.81,379.96,368.15,375.53,100413036


In [4]:
fig = go.Figure(data=[
    go.Candlestick(
        x=data["Date"],
        open=data["Open"],
        high=data["High"],
        low=data["Low"],
        close=data["Close"],
        name="TSLA",

        increasing_line_color="green",
        decreasing_line_color="red",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Open: $%{open:.2f}<br>"
            "High: $%{high:.2f}<br>"
            "Low: $%{low:.2f}<br>"
            "Close: $%{close:.2f}"
            "<extra></extra>"
        )
    )
])

fig.update_layout(
    title="TSLA Stock Price – Interactive Candlestick Chart",
    xaxis_title="Date",
    yaxis_title="Stock Price (USD)",
    template="plotly_white",
    height=650,
    xaxis_rangeslider_visible=True,
    hovermode="x unified"
)

fig.show()

In [5]:
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=data["Date"],
    open=data["Open"],
    high=data["High"],
    low=data["Low"],
    close=data["Close"],
    name="TSLA Price"
))

fig.add_trace(go.Bar(
    x=data["Date"],
    y=data["Volume"],
    name="Trading Volume",
    yaxis="y2",
    opacity=0.3
))

fig.update_layout(
    title="TSLA Price and Trading Volume",
    xaxis_title="Date",
    yaxis_title="Stock Price (USD)",
    yaxis2=dict(
        title="Volume",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    template="plotly_white",
    height=700,
    xaxis_rangeslider_visible=True
)

fig.show()

In [7]:
data["MA_7"] = data["Close"].rolling(7).mean()
data["MA_30"] = data["Close"].rolling(30).mean()

fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=data["Date"],
    open=data["Open"],
    high=data["High"],
    low=data["Low"],
    close=data["Close"],
    name="Candlestick"
))

fig.add_trace(go.Scatter(
    x=data["Date"],
    y=data["MA_7"],
    mode="lines",
    name="7-Period Moving Average"
))

fig.add_trace(go.Scatter(
    x=data["Date"],
    y=data["MA_30"],
    mode="lines",
    name="30-Period Moving Average"
))

fig.update_layout(
    title="TSLA Candlestick Chart with Moving Averages",
    xaxis_title="Date",
    yaxis_title="Stock Price (USD)",
    template="plotly_white",
    height=650,
    xaxis_rangeslider_visible=True
)

fig.show()

In [8]:
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=data["Date"],
    open=data["Open"],
    high=data["High"],
    low=data["Low"],
    close=data["Close"],
    name="TSLA"
))

fig.update_layout(
    title="TSLA Stock Price – Date Range Analysis",
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=7, label="7 Records", step="all"),
                dict(count=30, label="30 Records", step="all"),
                dict(step="all", label="All Data")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    ),
    yaxis_title="Stock Price (USD)",
    template="plotly_white",
    height=650
)

fig.show()

In [9]:
print("Total Records:", len(data))
print("Date Range:", data["Date"].min(), "to", data["Date"].max())
print("Missing Values:", data[required_columns].isnull().sum().sum())
print("Duplicate Dates:", data["Date"].duplicated().sum())

print("\nCandlestick visualization completed!")

Total Records: 95
Date Range: 2025-01-02 00:00:00 to 2025-05-14 00:00:00
Missing Values: 0
Duplicate Dates: 0

Candlestick visualization completed!
